# RL Cinematic Agent Pipeline — Kaggle Test Notebook
Tests the fully local `agent.py` pipeline:
`RLRunMemory` → `LocalLLM` → `DirectorAgent` (PPO) → `PromptEnricher` → `SelfCritic` (CLIP) → `run_rl_pipeline`

All LLM inference and PPO training run locally on a HuggingFace model — no external API calls.


## 1) Install Dependencies
Install packages required by `agent.py` (transformers, TRL for PPO, open_clip for SelfCritic, etc.).


In [ ]:
!pip -q install --upgrade pip
# Core ML / LLM
!pip -q install torch transformers accelerate
# TRL — pin to 0.8.6 (last release with the old PPO API used by agent.py:
#   AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer).
#   TRL >= 0.9 moved / removed these from the top-level namespace.
!pip -q install "trl==0.8.6"
# CLIP-based SelfCritic
!pip -q install open_clip_torch
# Video frame sampling
!pip -q install opencv-python-headless
# General utilities
!pip -q install requests scikit-learn pandas numpy
print("All dependencies installed.")


## 2) Clone Repository and Set Working Directory


In [ ]:
import os
from pathlib import Path

WORK_DIR  = Path("/kaggle/working")
CLONE_DIR = WORK_DIR / "tonmoy99_create-RL-Brain"

if not CLONE_DIR.exists():
    !git clone https://github.com/Tonmoy221/tonmoy99_create-RL-Brain.git "{CLONE_DIR.name}"

search_roots = [
    WORK_DIR / "tonmoy99_Vedio-Gen",
    CLONE_DIR,
]

matches = []
for root in search_roots:
    if root.exists():
        matches.extend(root.rglob("agent.py"))

if not matches:
    raise FileNotFoundError(
        "agent.py not found under /kaggle/working. "
        "Expected in your project folder after clone."
    )

# Prefer the intended project folder if multiple matches exist.
matches = sorted(
    matches,
    key=lambda p: (
        "tonmoy99_Vedio-Gen" not in str(p.parent),
        len(str(p.parent)),
    ),
)

script_path = matches[0]
repo_path   = script_path.parent

os.chdir(repo_path)
print("Working directory :", os.getcwd())
print("Using agent.py    :", script_path)
print("agent.py exists   :", Path("agent.py").exists())
print("config.py exists  :", Path("config.py").exists())


## 3) Configure Local Model and Seed Prompt
Set the HuggingFace model (`LocalLLM`) and the cinematic seed prompt.
No API keys are needed — all inference runs on-device.


In [ ]:
import torch

# ── Local HuggingFace model used for both LLM generation and PPO policy ──────
# Examples:
#   "meta-llama/Meta-Llama-3-8B-Instruct"   (requires HF token + >16 GB VRAM)
#   "mistralai/Mistral-7B-Instruct-v0.3"     (open weights, ~14 GB VRAM)
#   "Qwen/Qwen2.5-7B-Instruct"              (open weights, ~14 GB VRAM)
#   "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   (lightweight, for smoke-testing)
LOCAL_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ── Seed prompt ───────────────────────────────────────────────────────────────
SEED_PROMPT = (
    "A lone astronaut drifts through the crystalline rings of Saturn, "
    "helmet visor reflecting a dying sun."
)

# ── Output root ───────────────────────────────────────────────────────────────
OUTPUT_ROOT = "."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device          : {DEVICE}")
print(f"Local model     : {LOCAL_MODEL_NAME}")
print(f"Seed prompt     : {SEED_PROMPT[:80]}…")


## 4) Smoke-test: Import agent.py Components
Verify that all classes — `RLRunMemory`, `LocalLLM`, `DirectorAgent`, `PromptEnricher`, `SelfCritic`, `run_rl_pipeline` — import correctly.


In [ ]:
import sys, os

repo_root = None
for candidate in [
    "/kaggle/working/tonmoy99_create-RL-Brain",
    "/kaggle/working",
]:
    p = Path(candidate)
    if (p / "agent.py").exists():
        repo_root = p
        break
if repo_root is None:
    for hit in Path("/kaggle/working").rglob("agent.py"):
        repo_root = hit.parent
        break
if repo_root is None:
    raise FileNotFoundError("Cannot find agent.py under /kaggle/working")

os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import all public symbols from agent.py
from agent import (
    RLRunMemory,
    LocalLLM,
    DirectorAgent,
    PromptEnricher,
    SelfCritic,
    CritiqueResult,
    run_rl_pipeline,
)

print("PASS: All agent.py components imported successfully.")
print("  RLRunMemory  :", RLRunMemory)
print("  LocalLLM     :", LocalLLM)
print("  DirectorAgent:", DirectorAgent)
print("  PromptEnricher:", PromptEnricher)
print("  SelfCritic   :", SelfCritic)
print("  run_rl_pipeline:", run_rl_pipeline)


## 5) Verify RLRunMemory — Character Appearance Registry
Unit-test `RLRunMemory`: save/load cycle with full physical character appearance fields and reward curve.


In [ ]:
import json, os, tempfile

tmp_dir  = tempfile.mkdtemp()
mem_path = os.path.join(tmp_dir, "memory_rl", "state_rl.json")

mem = RLRunMemory(state_path=mem_path)

# Register a character with full physical appearance
mem.character_registry["char_1"] = {
    "name":                    "Elena Vasquez",
    "gender":                  "female",
    "age":                     "32",
    "skin_tone":               "medium olive",
    "body_type":               "athletic",
    "height":                  "5ft 7in",
    "hair_color":              "jet black",
    "hair_style":              "long wavy",
    "eye_color":               "dark brown",
    "facial_expression":       "determined",
    "distinguishing_features": "small scar above left eyebrow",
    "wardrobe":                "worn leather jacket, dark cargo pants, scuffed boots",
    "visual_description":      "Athletic olive-skinned woman with long wavy black hair, "
                               "determined dark-brown eyes, and a leather jacket.",
}
mem.location_registry["loc_1"] = {
    "name":             "Saturn Ring Observation Deck",
    "material_palette": "metallic silver, deep black, crystalline blue-white",
}
mem.reward_curve.extend([0.45, 0.61, 0.73, 0.80])
mem.save()

# Round-trip: load and verify
mem2 = RLRunMemory(state_path=mem_path)
assert mem2.load(), "RLRunMemory.load() returned False"
assert "char_1" in mem2.character_registry
char = mem2.character_registry["char_1"]
assert char["skin_tone"]   == "medium olive", f"Wrong skin_tone: {char['skin_tone']}"
assert char["body_type"]   == "athletic"
assert char["hair_color"]  == "jet black"
assert char["eye_color"]   == "dark brown"
assert len(mem2.reward_curve) == 4
print("PASS: RLRunMemory save/load cycle.")
print("Reward curve:", mem2.reward_curve)
print("\nState summary:\n", mem2.get_state_for_agent())


## 6) Smoke-test LocalLLM
Load the local HuggingFace model and verify it can generate a response.
This is the same model used by `DirectorAgent` for `generate_creative_document`, `refine_scene_prompt`, and `critique_scene_prompt`.


In [ ]:
local_llm = LocalLLM(
    model_name_or_path=LOCAL_MODEL_NAME,
    device=DEVICE,
    max_new_tokens=256,
    temperature=0.4,
)

smoke_response = local_llm(
    system_prompt="You are an expert cinematic director AI. Be concise.",
    user_prompt=(
        "In 1-2 sentences, describe the visual appearance of a female astronaut character "
        "with olive skin, athletic build, and jet-black hair for a scene set among Saturn's rings."
    ),
)
print("LocalLLM smoke-test response:\n")
print(smoke_response)
print("\nPASS: LocalLLM is working correctly.")


## 7) Smoke-test DirectorAgent (LLM + PPO)
Build a `DirectorAgent` with the loaded `LocalLLM`.
Test `generate_creative_document` (with per-character physical appearance), `refine_scene_prompt` (numbered aliases), `critique_scene_prompt`, and the `update_policy` PPO step.


In [ ]:
import traceback

agent = DirectorAgent(local_llm=local_llm, device=DEVICE)
print("DirectorAgent initialized with PPO policy.")

# ── generate_creative_document ───────────────────────────────────────────────
print("\n[STEP] generate_creative_document …")
try:
    creative_doc = agent.generate_creative_document(
        seed_prompt=SEED_PROMPT,
        memory_state="RLRunMemory State\nCharacters: None\nLocations: None\nRecent continuity: None",
    )
    print(f"  Characters : {[c.get('name') for c in creative_doc.get('characters', [])]}")
    print(f"  Scenes     : {[s.get('scene_id') for s in creative_doc.get('scenes', [])]}")
    # Verify physical appearance fields are present on every character
    for ch in creative_doc.get("characters", []):
        for field in ["skin_tone", "body_type", "height", "hair_color",
                      "hair_style", "eye_color", "facial_expression", "distinguishing_features"]:
            assert field in ch, f"Character '{ch.get('name')}' missing '{field}'"
    print("  PASS: All character appearance fields present.")
except Exception as exc:
    print(f"  WARN: generate_creative_document raised: {exc}")
    traceback.print_exc()
    # Build a minimal fallback for downstream tests
    creative_doc = {
        "characters": [{
            "id": "char_1", "name": "Elena Vasquez", "age": "32", "gender": "female",
            "skin_tone": "medium olive", "body_type": "athletic", "height": "5ft 7in",
            "hair_color": "jet black", "hair_style": "long wavy", "eye_color": "dark brown",
            "facial_expression": "determined", "distinguishing_features": "scar above left eyebrow",
            "wardrobe": "worn leather jacket, dark cargo pants", "visual_description":
            "Athletic olive-skinned woman with long wavy black hair.", "reference_image_paths": [],
        }],
        "locations": [{
            "id": "loc_1", "name": "Saturn Ring Deck", "scenery_type": "space station exterior",
            "material_palette": "metallic silver, crystalline blue-white",
            "visual_description": "A gleaming observation platform among Saturn's icy rings.",
            "reference_image_paths": [],
        }],
        "scenes": [{
            "scene_id": "scene_1",
            "narrative_description": "Elena floats near the viewport, gazing at the rings.",
            "scene_prompt": "An astronaut watches Saturn's rings through a panoramic visor.",
            "character_ids": ["char_1"], "location_id": "loc_1",
            "camera_style": "slow push-in", "emotional_tone": "awe",
            "time_of_day": "N/A (space)", "weather": "none",
            "continuity_constraints": "Keep Elena's appearance locked.",
        }],
        "cinematic_style": "cinematic realism", "narrative_arc": "setup -> wonder -> resolve",
        "color_grading": "filmic teal-orange",
    }
    print("  Using fallback creative_doc for downstream steps.")

# ── refine_scene_prompt (numbered character aliases) ─────────────────────────
print("\n[STEP] refine_scene_prompt …")
scene = creative_doc["scenes"][0]
try:
    refined = agent.refine_scene_prompt(
        scene=scene,
        creative_document=creative_doc,
        memory_state="RLRunMemory State\nCharacters: None\nLocations: None",
    )
    print(f"  Refined prompt (first 200 chars):\n  {refined[:200]}")
    print("  PASS: refine_scene_prompt returned output.")
except Exception as exc:
    print(f"  WARN: refine_scene_prompt raised: {exc}")
    refined = scene["scene_prompt"]

# ── critique_scene_prompt ────────────────────────────────────────────────────
print("\n[STEP] critique_scene_prompt …")
try:
    critique = agent.critique_scene_prompt(
        scene=scene,
        scene_prompt=refined,
        creative_document=creative_doc,
        memory_state="RLRunMemory State\nCharacters: None\nLocations: None",
    )
    print(f"  score={critique['score']:.4f}  accept={critique['accept']}")
    print(f"  issues={critique['issues']}")
    print("  PASS: critique_scene_prompt returned output.")
except Exception as exc:
    print(f"  WARN: critique_scene_prompt raised: {exc}")
    critique = {"score": 0.5, "accept": True, "issues": [], "revised_prompt": refined}

# ── update_policy (PPO step) ─────────────────────────────────────────────────
print("\n[STEP] update_policy (PPO) …")
ppo_result = agent.update_policy(
    scene_state_text="Scene 1: Elena Vasquez floats near the viewport.",
    action_prompt_text=refined,
    reward=float(critique["score"]),
)
print(f"  PPO status : {ppo_result.get('status')}")
print(f"  PPO stats  : {list(ppo_result.get('stats', {}).keys())[:5]}")
print("PASS: DirectorAgent smoke-test complete.")


## 8) Run the Full RL Pipeline
Runs `run_rl_pipeline` end-to-end:
1. Generates a Creative Document (characters with full physical appearance + locations + scenes).
2. Per-scene: `refine_scene_prompt` (with numbered aliases) → `critique_scene_prompt` → PPO `update_policy`.
3. Writes `creative_document_rl.json`, `scene_prompts_rl.json`, `rl_pipeline_report.json`, and `state_rl.json`.


In [ ]:
import traceback

try:
    report_path = run_rl_pipeline(
        seed_prompt=SEED_PROMPT,
        output_root=OUTPUT_ROOT,
        local_model_name=LOCAL_MODEL_NAME,
        resume=False,
    )
    print("Pipeline completed successfully.")
    print("Report path:", report_path)
except Exception as exc:
    print("Pipeline failed with exception:", str(exc))
    print("Detailed traceback:\n")
    traceback.print_exc()
    raise


## 9) Validate Pipeline Artifacts
Assert all expected output files exist and contain the required fields.


In [ ]:
import json
from pathlib import Path

creative_doc_path  = Path("story_bible/rl_only/creative_document_rl.json")
scene_prompts_path = Path("output/rl_only/scene_prompts_rl.json")
report_path_file   = Path("output/rl_only/rl_pipeline_report.json")
memory_path_file   = Path("memory_rl/state_rl.json")

required_paths = [creative_doc_path, scene_prompts_path, report_path_file, memory_path_file]
missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    raise FileNotFoundError(f"Missing expected artifacts: {missing}")

creative_doc   = json.loads(creative_doc_path.read_text(encoding="utf-8"))
scene_prompts  = json.loads(scene_prompts_path.read_text(encoding="utf-8"))
report         = json.loads(report_path_file.read_text(encoding="utf-8"))
memory_state   = json.loads(memory_path_file.read_text(encoding="utf-8"))

scene_count        = len(creative_doc.get("scenes", []))
mean_score         = float(report.get("mean_critique_score", 0.0))
mean_ppo_reward    = float(report.get("mean_ppo_reward", 0.0))
continuity_count   = len(memory_state.get("continuity_log", []))
reward_curve       = memory_state.get("reward_curve", [])

# Verify every character has physical appearance fields
appearance_fields = [
    "skin_tone", "body_type", "height", "hair_color", "hair_style",
    "eye_color", "facial_expression", "distinguishing_features",
]
for ch in creative_doc.get("characters", []):
    missing_fields = [f for f in appearance_fields if not ch.get(f)]
    if missing_fields:
        print(f"  WARN: Character '{ch.get('name')}' missing or empty: {missing_fields}")

# Verify per-scene character assignments are saved
for sp in scene_prompts:
    assert "scene_character_assignments" in sp, \
        f"scene '{sp.get('scene_id')}' missing scene_character_assignments"
    assert "ppo_reward" in sp, \
        f"scene '{sp.get('scene_id')}' missing ppo_reward"

print("PASS: All RL pipeline artifacts generated and validated.")
print(f"  scene_count          : {scene_count}")
print(f"  mean_critique_score  : {round(mean_score, 4)}")
print(f"  mean_ppo_reward      : {round(mean_ppo_reward, 4)}")
print(f"  continuity_log_count : {continuity_count}")
print(f"  reward_curve         : {reward_curve}")


## 10) Visualize Results
Plot the PPO reward curve across scenes. Display the character physical appearance registry and the scene-level character alias assignments.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ── Reward curve plot ────────────────────────────────────────────────────────
if reward_curve:
    plt.figure(figsize=(10, 4))
    plt.plot(reward_curve, marker="o", linewidth=2, label="PPO reward (per scene)")
    plt.axhline(y=mean_ppo_reward, color="red", linestyle="--", label=f"Mean={mean_ppo_reward:.4f}")
    plt.title("RL Pipeline — PPO Reward Curve Across Scenes")
    plt.xlabel("Scene index")
    plt.ylabel("PPO reward (= critique score)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No reward_curve data to plot.")

# ── Character physical appearance registry ───────────────────────────────────
char_rows = []
for ch in creative_doc.get("characters", []):
    char_rows.append({
        "id":           ch.get("id", ""),
        "name":         ch.get("name", ""),
        "gender":       ch.get("gender", ""),
        "age":          ch.get("age", ""),
        "skin_tone":    ch.get("skin_tone", ""),
        "body_type":    ch.get("body_type", ""),
        "height":       ch.get("height", ""),
        "hair":         f"{ch.get('hair_color', '')} / {ch.get('hair_style', '')}",
        "eye_color":    ch.get("eye_color", ""),
        "expression":   ch.get("facial_expression", ""),
        "features":     ch.get("distinguishing_features", ""),
        "wardrobe":     ch.get("wardrobe", "")[:60] + "…" if len(ch.get("wardrobe", "")) > 60 else ch.get("wardrobe", ""),
    })
char_df = pd.DataFrame(char_rows)

# ── Scene prompt + character aliases + PPO reward table ──────────────────────
scene_rows = []
for sp in scene_prompts:
    aliases = sp.get("scene_character_assignments", {})
    alias_str = ", ".join(f"{v}" for v in aliases.values()) if aliases else "—"
    scene_rows.append({
        "scene_id":          sp.get("scene_id"),
        "critique_score":    sp.get("critique", {}).get("score", 0.0),
        "ppo_reward":        sp.get("ppo_reward", 0.0),
        "ppo_status":        sp.get("ppo_update", {}).get("status", "—"),
        "character_aliases": alias_str,
        "prompt_preview":    sp.get("prompt", "")[:80] + "…",
    })
scene_df = pd.DataFrame(scene_rows)

print("\n── Character Physical Appearance Registry ──")
display(char_df)

print("\n── Scene Prompts | PPO Rewards | Character Aliases ──")
display(scene_df)


## 11) Smoke-test SelfCritic (CLIP-based Visual Consistency)
Instantiate `SelfCritic` and verify it loads the CLIP `ViT-B-32` model and returns a `CritiqueResult` gracefully even without a real video clip (reference images unavailable path).


In [ ]:
from agent import SelfCritic, CritiqueResult

print("[SelfCritic] Loading CLIP ViT-B-32 model …")
try:
    critic = SelfCritic(threshold=0.25, device=DEVICE)
    print("[SelfCritic] Model loaded successfully.")

    # Evaluate with a non-existent clip path — should return a graceful fallback
    result: CritiqueResult = critic.evaluate(
        clip_path="/tmp/nonexistent_clip.mp4",        # no real video needed for smoke test
        character_reference_images=[],                 # no reference images
        location_reference_images=[],
        current_prompt=scene_prompts[0].get("prompt", ""),
    )
    print(f"  consistency_score : {result.consistency_score}")
    print(f"  critique          : {result.critique}")
    print(f"  retry             : {result.retry}")
    print("PASS: SelfCritic smoke-test passed (graceful fallback on missing video).")
    critic.close()

except Exception as exc:
    print(f"WARN: SelfCritic raised: {exc}")
    print("This is expected if CLIP is not installed or VRAM is insufficient.")


## 12) Final Pass/Fail Summary


In [ ]:
checks = {
    "creative_document_rl.json exists":      creative_doc_path.exists(),
    "scene_prompts_rl.json exists":          scene_prompts_path.exists(),
    "rl_pipeline_report.json exists":        report_path_file.exists(),
    "memory state_rl.json exists":           memory_path_file.exists(),
    "characters have physical appearance":   all(
        ch.get("skin_tone") and ch.get("body_type") and ch.get("hair_color")
        for ch in creative_doc.get("characters", [])
    ),
    "scenes have character aliases":         all(
        "scene_character_assignments" in sp for sp in scene_prompts
    ),
    "scenes have ppo_reward":                all(
        "ppo_reward" in sp for sp in scene_prompts
    ),
    "reward_curve non-empty":                len(reward_curve) > 0,
    "mean_critique_score > 0":               mean_score > 0,
}

print("=" * 60)
print("FINAL PASS/FAIL REPORT — RL Agent Pipeline (agent.py)")
print("=" * 60)
all_pass = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  [{status}] {check}")

print("=" * 60)
print("Overall result:", "ALL PASS ✓" if all_pass else "SOME CHECKS FAILED ✗")
print(f"  scene_count         = {scene_count}")
print(f"  mean_critique_score = {mean_score:.4f}")
print(f"  mean_ppo_reward     = {mean_ppo_reward:.4f}")
print(f"  reward_curve        = {reward_curve}")
print("=" * 60)
